# 07 · Baselines

| Baseline | Idea | Notes |
|---|---|---|
| **TF-IDF** | rank sentences by cosine similarity to the TF-IDF centroid | fast, no training |
| **TextRank** | PageRank over a sentence-similarity graph (NumPy) | no networkx dependency |
| **LSTM seq2seq** | tiny GRU/LSTM encoder-decoder + Bahdanau attention | academic baseline only; small vocab, no pretraining, short inputs |

The LSTM trains in minutes on free CPU/GPU with `scripts/train_lstm_baseline.py`.

In [ ]:
print('[1/3] Extractive baselines on the synthetic sample')
import sys; sys.path.insert(0, '.')
from src.data_loader import load_transcript_file
from src.baselines import tfidf_summarize, text_rank_summarize
m = load_transcript_file('data/samples/sample_meeting.txt')
text = chr(10).join(f"{t['speaker']}: {t['text']}" for t in m['turns'])
print('TF-IDF    :', tfidf_summarize(text, 3))
print()
print('TextRank  :', text_rank_summarize(text, 3))

In [ ]:
print('[2/3] LSTM baseline: train if no checkpoint (tiny, minutes on CPU)')
from pathlib import Path
from src.config import load_config, resolve_path
from src.utils import setup_logging
cfg = load_config(); log = setup_logging(cfg, 'nb07_lstm')
ckpt = resolve_path(cfg, 'models') / 'lstm_baseline.pt'
if ckpt.exists():
    print('checkpoint exists ->', ckpt)
else:
    !python -u scripts/train_lstm_baseline.py --pairs 40 --epochs 3

In [ ]:
print('[3/3] Generate an LSTM summary and compare with the extractive ones')
from src.baselines import LSTMBaseline
base = LSTMBaseline.load(ckpt)
print('LSTM      :', base.summarize(text)[:600])